In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC #### Install MLCORE SDK

# COMMAND ----------

# MAGIC %pip install google-auth
# MAGIC %pip install google-cloud-storage
# MAGIC %pip install azure-storage-blob
# MAGIC %pip install azure-identity
# MAGIC # %pip install protobuf==3.17.2
# MAGIC %pip install sparkmeasure

# COMMAND ----------

# MAGIC %md
# MAGIC #### Install Deep Checks, MLFlow, Pandas and Numpy to specific version

# COMMAND ----------

# MAGIC %pip install deepchecks
# MAGIC # %pip install numpy==1.19.1
# MAGIC # %pip install pandas==1.0.5
# MAGIC # %pip install matplotlib==3.3.2
# MAGIC %pip install numpy==1.23.5
# MAGIC %pip install databricks-sql-connector

# COMMAND ----------

dbutils.library.restartPython()

# COMMAND ----------

# MAGIC %md
# MAGIC #### Import libraries

# COMMAND ----------

from sparkmeasure import StageMetrics
from sparkmeasure import TaskMetrics

taskmetrics = TaskMetrics(spark)
stagemetrics = StageMetrics(spark)

taskmetrics.begin()
stagemetrics.begin()

# COMMAND ----------

from deepchecks.tabular import Dataset
from deepchecks.tabular import Suite
import importlib
import time
from pyspark.sql import functions as F
import numpy as np
import mlflow
import json
from tenacity import retry, stop_after_delay, stop_after_attempt, wait_exponential
import requests
from requests.structures import CaseInsensitiveDict
from utils import utils
from utils import uc_utils
from databricks import sql
# Disable logs from spark.
import logging

logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from utils.secret_mapping import SECRET_MAPPING, CONFIGS
from utils.vault_scope import VAULT_SCOPE

# Disable auto-logging of runs in the mlflow
import mlflow

mlflow.autolog(disable=True)

# COMMAND ----------

# MAGIC %md
# MAGIC #### Read inputs parameters provided to the notebook.

# COMMAND ----------

model_info = json.loads(dbutils.widgets.get("model_info"))
project_id = model_info.get("project_id", None)
project_name = model_info.get("project_name", None)
model_name = model_info.get("model_name", None)
model_version = model_info.get("model_version", None)
model_data_path = model_info.get("model_data_path", None)

if isinstance(model_info["feature_columns"], str):
    try:
        feature_columns = json.loads(model_info["feature_columns"])
    except:
        feature_columns = json.loads(model_info["feature_columns"].replace("'", '"'))

if isinstance(model_info["feature_columns"], str):
    try:
        target_columns = json.loads(model_info.get("target_columns", []))
    except:
        target_columns = json.loads(model_info["target_columns"].replace("'", '"'))

media_artifacts_path = model_info.get("media_artifact_path", None)
model_artifact_id = model_info.get("model_artifact_id", None)
test_scope = model_info.get("scope", "fulltest")

# COMMAND ----------

with open('../data_config/TestConfig.json', 'r') as file:
    test_config = json.load(file)

# COMMAND ----------

# MAGIC %md
# MAGIC #### Utility Methods

# COMMAND ----------


def take_random_rows(df, n=20000):
    """
    Samples n rows randomly from the input dataframe
    """
    fraction_value = n / df.count()
    if fraction_value > 1:
        return df
    else:
        return df.sample(withReplacement=False, fraction=fraction_value, seed=2023)


def detect_categorical_cols(df, threshold=5):
    """
    Get the Categorical columns with greater than threshold percentage of unique values.

    This function returns the Categorical columns with the unique values in the column
    greater than the threshold percentage.

    Parameters
    ----------
    df: pyspark.sql.DataFrame
    threshold : int , default = 5
        threshold value in percentage

    Returns
    -------
    report_data : dict
        dictionary containing the Numeric column data.

    """
    df = df.toPandas()
    no_of_rows = df.shape[0]
    possible_cat_cols = (
        df.convert_dtypes()
        .select_dtypes(exclude=[np.datetime64, "float", "float64"])
        .columns.values.tolist()
    )
    temp_series = df[possible_cat_cols].apply(
        lambda col: (len(col.unique()) / no_of_rows) * 100 > threshold
    )
    cat_cols = temp_series[temp_series == False].index.tolist()
    return cat_cols


def get_env_vault_scope():
    """
    Returns env and vault scope
    """
    try:
        env = dbutils.widgets.get("deployment_env")
    except:
        env = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        ).split("/")[2]

    return env, VAULT_SCOPE.get(env, {}).get("client_name", "")


def get_access_tokens(client_id, scope, client_secret, vault_scope):
    """
    Returns a bearer token
    """

    headers = CaseInsensitiveDict()
    headers["Content-Type"] = "application/x-www-form-urlencoded"
    data = {}
    data["client_id"] = client_id
    data["grant_type"] = "client_credentials"
    data["scope"] = scope
    data["client_secret"] = client_secret
    tenant_id = secrets_object.get("az-directory-tenant", "")
    url = "https://login.microsoftonline.com/" + tenant_id + "/oauth2/v2.0/token"
    resp = requests.post(url, headers=headers, data=data).json()
    token = resp["access_token"]
    token_string = "Bearer" + " " + token
    return token_string


def get_app_url():
    """
    Returns env and vault scope
    """
    print("Fetching API_ENDPOINT from secrets.")
    env, vault_scope = get_env_vault_scope()
    API_ENDPOINT = ""
    if env in ["dev", "qa"]:
        API_ENDPOINT = (
            secrets_object.get(f"az-app-service-{env}-url-2", "") + "/"
        )
    else:
        API_ENDPOINT = (
            secrets_object.get(f"az-app-service-url", "") + "/"
        )
    return API_ENDPOINT


def get_headers(vault_scope):
    """
    Returns API headers
    """
    h1 = CaseInsensitiveDict()
    client_id = secrets_object.get("az-api-client-id", "")
    scope = client_id + "/.default"
    client_secret = secrets_object.get("az-api-client-secret", "")
    h1["Authorization"] = get_access_tokens(
        client_id, scope, client_secret, vault_scope
    )
    h1["Content-Type"] = "application/json"
    return h1


def fetch_secrets_from_dbutils(dbutils, message_logs=[]):
    _, vault_scope = get_env_vault_scope()
    secrets_object = {}
    for secret_key, secret_value in SECRET_MAPPING.items():
        try:
            secrets_object[secret_key] = dbutils.secrets.get(
                scope=vault_scope, key=secret_value
            )
        except Exception as e:
            utils.log(
                f"Error fetching secret for '{secret_value} using dbutils': {e}",
                message_logs,
                "warning",
            )

            # fetch the secret from config file
            secrets_object[secret_key] = CONFIGS.get(secret_value, "")

    return secrets_object



def get_cluster_info(spark):
    p = "spark.databricks.clusterUsageTags."
    conf = spark.sparkContext.getConf().getAll()
    conf_dict = {k.replace(p, ""): v for k, v in conf if k.startswith(p)}

    return conf_dict


@retry(
    wait=wait_exponential(min=4, multiplier=1, max=10),
    stop=(stop_after_delay(40) | stop_after_attempt(5)),
)
def task_log_data():

    ts = int(time.time() * 1000000)

    task_log_data = {
        "project_id": project_id,
        "version": version,
        "job_id": str(job_id),
        "run_id": str(run_id),
        "task_id": str(task_run_id),
        "status": "running",
        "start_time": str(ts),
        "model_train_session_id": model_artifact_details.get("model_train_session_id", ""),
        "created_by_id": model_artifact_details.get("created_by_id", ""),
        "created_by_name": model_artifact_details.get("created_by_name", ""),
        "job_type": "Model_Testing",
    }
    h1 = get_headers(vault_scope)
    response = requests.post(
        API_ENDPOINT + JOB_TASK_ADD, json=task_log_data, headers=h1
    )
    utils.log(
        f"\n\
    Logging task:\n\
    endpoint - {JOB_TASK_ADD}\n\
    payload  - {task_log_data}\n\
    response - {response}\n",
        message_run,
    )

    t = str(int(time.time() * 1000000))

    return message_run, message_task


@retry(
    wait=wait_exponential(min=4, multiplier=1, max=10),
    stop=(stop_after_delay(40) | stop_after_attempt(5)),
)
def job_task_update(status):

    ts = str(int(time.time() * 1000000))
    task_log_data = {
        "job_id": str(job_id),
        "run_id": str(run_id),
        "task_id": str(task_run_id),
        "end_time": str(ts),
        "status": status,
        "message": message_task,
        "updated_by_id": model_artifact_details.get("created_by_id", ""),
        "updated_by_name": model_artifact_details.get("created_by_name", ""),
        "job_type": "Model_Testing",
        "run_notebook_url": run_notebook_url,
    }
    h1 = get_headers(vault_scope)
    response = requests.put(
        API_ENDPOINT + JOB_TASK_UPDATE, json=task_log_data, headers=h1
    )
    utils.log(
        f"\n\
    Logging task '{status}': \n\
    endpoint - {JOB_TASK_UPDATE}\n\
    payload  - {task_log_data}\n\
    response - {response}\n",
        message_run,
    )

    if response.status_code not in [200, 201]:
        raise Exception(
            f"API Error : The {JOB_TASK_UPDATE} API returned {response.status_code} response."
        )

    return message_run, message_task

def job_runs_update(status, message):
    stagemetrics.end()
    taskmetrics.end()
    stagemetrics.print_report()
    stage_Df = stagemetrics.create_stagemetrics_DF("PerfStageMetrics")
    task_Df = taskmetrics.create_taskmetrics_DF("PerfTaskMetrics")
    aggregate_compute_metrics = (
        stagemetrics.aggregate_stagemetrics_DF()
        .select("executorCpuTime", "peakExecutionMemory")
        .collect()[0]
        .asDict()
    )
    aggregate_compute_metrics["executorCpuTime"] = (
        aggregate_compute_metrics["executorCpuTime"] / 1000
        if aggregate_compute_metrics["executorCpuTime"]
        else 0
    )
    aggregate_compute_metrics["peakExecutionMemory"] = (
        aggregate_compute_metrics["peakExecutionMemory"] / (1024 * 1024)
        if aggregate_compute_metrics["peakExecutionMemory"]
        else 0
    )

    compute_metrics = {
        "stagemetrics": stage_Df.rdd.map(lambda row: row.asDict()).collect(),
        "taskmetrics": task_Df.rdd.map(lambda row: row.asDict()).collect(),
    }

    ts = str(int(time.time() * 1000000))
    message_run.append(
        {
            "time": ts,
            "message": "Job with " + str(job_id) + " and " + str(run_id) + message,
        }
    )

    log_data = {
        "project_id": model_artifact_details.get("project_id", ""),
        "version": str(model_artifact_details.get("version", "")),
        "job_id": str(job_id),
        "run_id": str(run_id),
        "end_time": str(ts),
        "status": status,
        "model_artifact_id": model_artifact_id,
        "message": message_run,
        "updated_by_id": model_artifact_details.get("created_by_id", ""),
        "updated_by_name": model_artifact_details.get("created_by_name", ""),
        "task_id": str(task_run_id),
        "job_type": "Model_Testing",
        "cpu": str(aggregate_compute_metrics.get("executorCpuTime", "NA")),
        "ram": str(aggregate_compute_metrics.get("peakExecutionMemory", "NA")),
        "cluster_info": get_cluster_info(spark),
        "compute_metrics": compute_metrics,
        "run_notebook_url": run_notebook_url,
        "media_artifacts_path": (
            report_directory.split("/Test_Validation")[0]
            if "report_directory" in globals()
            else ""
        ),
    }

    # If the job was successful, set task_update property to True to terminate the underlying tasks as well.
    if status.lower() == "success":
        log_data["tasks_update"] = True
    # Log the payload excluding large fields
    log_payload = {
        key: value
        for key, value in log_data.items()
        if key not in ["compute_metrics", "cluster_info", "message"]
    }
    print(f"API URL : {JOB_RUNS_UPDATE}. Payload : {log_payload}")

    response = requests.put(API_ENDPOINT + JOB_RUNS_UPDATE, json=log_data, headers=h1)

    print(
        f"\n\
    endpoint - {JOB_RUNS_UPDATE}\n\
    response - {response}\n"
    )

    utils.save_sparkmeasure_aggregated_tables(
        project_name=project_name,
        job_run_update_payload=log_data,
        compute_usage_metrics=aggregate_compute_metrics,
        stagemetrics=stagemetrics,
        taskmetrics=taskmetrics,
        api_endpoint=API_ENDPOINT,
        headers=h1,
        spark=spark,
        dbutils=dbutils,
    )


@retry(
    wait=wait_exponential(min=4, multiplier=1, max=10),
    stop=(stop_after_delay(40) | stop_after_attempt(5)),
)
def get_model_artifacts(artifact_id):
    ARTIFACTS_ENDPOINT = f"mlapi/modelartifact/get?model_artifact_id={artifact_id}"
    h1 = get_headers(vault_scope)
    response = requests.get(API_ENDPOINT + ARTIFACTS_ENDPOINT, headers=h1)
    if response.status_code not in [200, 201]:
        raise Exception(
            f"API Error : The get_model_artifacts API returned {response.status_code} response."
        )

    utils.log(
        f"\n\
    endpoint - {ARTIFACTS_ENDPOINT}\n\
    response - {response}",
        message_run,
    )
    return response


def push_model_info(model_info, model_artifact_id, model_test_status):
    data = {
        "project_id": model_info.get("project_id", ""),
        "version": str(model_info.get("version", "")),
        "job_id": str(model_info.get("job_id", "")),
        "run_id": str(model_info.get("run_id", "")),
        "model_name": model_info.get("model_name", ""),
        "model_id": model_info.get("model_id", ""),
        "metrics": model_info.get("metrics", ""),
        "train_metrics": model_info.get("train_metrics", ""),
        "mlflow_run_id": model_info.get("mlflow_run_id", ""),
        "mlflow_model_id": model_info.get("mlflow_model_id", ""),
        "transformed_table_id": str(model_info.get("transformed_table_id", "")),
        "ground_truth_table_id": str(model_info.get("ground_truth_table_id", "")),
        "transformed_table_name": model_info.get("transformed_table_name", ""),
        "parent_transform_job_id": model_info.get("parent_transform_job_id", ""),
        "parent_transform_run_id": str(model_info.get("parent_transform_run_id", "")),
        "model_train_session_id": str(model_info.get("model_train_session_id", "")),
        "created_by_id": model_info.get("created_by_id", ""),
        "created_by_name": model_info.get("created_by_name", ""),
        "algorithm_name": model_info.get("algorithm_name", ""),
        "model_version": model_info.get("model_version", ""),
        "model_runtime_env_id": model_info.get("model_runtime_env_id", ""),
        "model_evaluation_report_status": model_info.get(
            "model_evaluation_report_status", ""
        ),
        "model_train_variables": model_info.get("model_train_variables", ""),
        "feature_columns": model_info.get("feature_columns", ""),
        "target_columns": model_info.get("target_columns", ""),
        "status": model_info.get("status", ""),
        "partition_keys": model_info.get("partition_keys", ""),
        "partition_columns": model_info.get("partition_columns", ""),
        "features_metadata": model_info.get("features_metadata", {}),
        "session_index": model_info.get("session_index", ""),
        "run_notebook_url": model_info.get("run_notebook_url"),
        "parent_ground_truth_dit_job_id": model_info.get(
            "parent_ground_truth_dit_job_id", ""
        ),
        "parent_ground_truth_dit_run_id": model_info.get(
            "parent_ground_truth_dit_run_id", ""
        ),
        "model_uuid": model_info.get("model_uuid", ""),
        "report_id": "",
        "report_url": "",  # Deprecate
        "model_artifact_id": model_artifact_id,
        "train_data_date_list": model_info.get("train_data_date_list", ""),
        "output_table_id": model_info.get("output_table_id", ""),
        "model_train_output_table_id": model_info.get(
            "model_train_output_table_id", ""
        ),
        "tuning_trials": model_info.get("tuning_trials", ""),
        "model_test_status": model_test_status,
        "parent_pipeline_params": model_info.get("parent_pipeline_params", {}),
        "workspace_id":  model_info.get("workspace_id", ""),
        "tracking_env": model_info.get("tracking_env", ""),
        "model_flavor": model_info.get("model_flavor", "sklearn")
    }

    h1 = get_headers(vault_scope)

    response = requests.post(API_ENDPOINT + MODELARTIFACTS_ADD, json=data, headers=h1)
    if response.status_code not in [200, 201]:
        raise Exception(
            f"API Error : The {MODELARTIFACTS_ADD} API returned {response.status_code} response."
        )

    utils.log(
        f"\n\
    endpoint - {MODELARTIFACTS_ADD}\n\
    payload  - {data}\n\
    response - {response}",
        message_run,
    )


def generate_run_notebook_url(job_id, run_id):
    """
    Generates the databricks job run notebook url in runtime
    """
    workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
    workspace_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterOwnerOrgId")
    run_notebook_url = f"{workspace_url}/?o={workspace_id}#job/{job_id}/run/{run_id}"
    return run_notebook_url


def get_catalog_details(env):
    h1 = get_headers(vault_scope)
    response = requests.get(
        API_ENDPOINT + GET_CATALOG + env, headers=h1
    )
    print(
        f"\n\
    Logging task:\n\
    endpoint - {GET_CATALOG}\n\
    status   - {response}\n\
    response - {response.text}"
    )

    return response


# GLOBAL VARIABLES
message_run = []
message_task = []

env, vault_scope = get_env_vault_scope()
secrets_object = fetch_secrets_from_dbutils(dbutils, message_run)
deployment_env = dbutils.widgets.get("deployment_env")

JOB_TASK_ADD = "mlapi/job/task/log/add"
JOB_TASK_UPDATE = "mlapi/job/task/log/update"
JOB_RUNS_UPDATE = "mlapi/job/runs/log/update"
MODELARTIFACTS_ADD = "mlapi/modelartifacts/add"
MEDIA_ARTIFACTS_ADD = "mlapi/add_media_artifacts"
GET_CATALOG = "mlapi/get_catalog?deployment_env="

try:
    API_ENDPOINT = dbutils.widgets.get("tracking_base_url")
except:
    API_ENDPOINT = get_app_url()
h1 = get_headers(vault_scope)

# Job Parameters
job_id, run_id, task_run_id, taskKey = utils.get_job_details(dbutils)
run_notebook_url = generate_run_notebook_url(job_id, run_id)

az_container_name = secrets_object.get("az-container-name", "")
quota_project_id = secrets_object.get("gcp-api-quota-project-id", "")
az_storage_account = secrets_object.get("az-storage-account", "")
uc_container_name = secrets_object.get("uc-container-name", "")
uc_volume_name = secrets_object.get("uc-volume-name", "")

try:
    datalake_env = dbutils.widgets.get("datalake_env")
except Exception as e:
    utils.log(f"Exception while retrieving data lake environment : {e}", message_run)
    datalake_env = "delta"
utils.log(f"Data Lake Environment : {datalake_env}", message_run)

try:
    cloud_provider = dbutils.widgets.get("cloud_provider").lower()
except:
    cloud_provider = "gcp"
utils.log(f"Cloud Provider : {cloud_provider}", message_run)

# COMMAND ----------

from tenacity import retry, stop_after_delay, stop_after_attempt, wait_exponential


def get_gcp_auth_credentials(dbutils):
    import google.auth

    _, vault_scope = get_env_vault_scope()
    client_id = secrets_object.get("gcp-api-client-id", "")
    client_secret = secrets_object.get("gcp-api-client-secret", "")
    quota_project_id = secrets_object.get("gcp-api-quota-project-id", "")
    refresh_token = secrets_object.get("gcp-api-refresh-token", "")
    cred_dict = {
        "client_id": client_id,
        "client_secret": client_secret,
        "quota_project_id": quota_project_id,
        "refresh_token": refresh_token,
        "type": "authorized_user",
    }

    credentials, _ = google.auth.load_credentials_from_dict(cred_dict)

    return credentials


def __upload_blob_to_azure(
    dbutils, source_path="", file=None, target_path="", container_name="",
):
    from azure.identity import ClientSecretCredential
    from azure.storage.blob import BlobServiceClient

    try:
        TENANT_ID = secrets_object.get("az-directory-tenant", "")
        CLIENT_ID = secrets_object.get(f"az-api-client-id", "")
        CLIENT_SECRET = secrets_object.get(f"az-api-client-secret", "")
        STORAGE_ACCOUNT = secrets_object.get(f"az-storage-account", "")

        # Authentication Blob client
        credentials = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
        service_client = BlobServiceClient(
            f"https://{STORAGE_ACCOUNT}.blob.core.windows.net", credential=credentials
        )
        container_client = service_client.get_container_client(container_name)

        if source_path:
            # Upload from local source file path to target path in Azure Blob Storage
            with open(source_path, "rb") as data:
                container_client.upload_blob(name=target_path, data=data)
        elif file:
            # Upload from in-memory file to target path in Azure Blob Storage
            container_client.upload_blob(name=target_path, data=file)

    except Exception as e:
        print(f"Error came while uploading blob object to Azure: {e}")
        raise e


def __upload_blob_to_gcp(dbutils, container_name, source_path, target_path):
    from google.cloud import storage

    try:
        credentials = get_gcp_auth_credentials(dbutils)
        project = secrets_object.get( "gcp-api-quota-project-id", "")

        # Use the obtained credentials to create a client to interact with GCP services
        storage_client = storage.Client(credentials=credentials, project=project)

        bucket_client = storage_client.bucket(container_name)

        # Upload the model file to GCS
        blob = bucket_client.blob(target_path)
        blob.upload_from_filename(source_path)

    except Exception as e:
        print(f"Error came while uploading blob object from gcp : {e}")
        raise e


def upload_blob_to_cloud(**kwargs):
    """
    Upload the blob from the cloud storage.

    This function will help upload the blob from the cloud storage service like Azure, AWS, GCP.

    Parameters
    ----------
    **kwargs : dict
        Keyword arguments containing operation details, including the `resource_type`.

    Returns
    -------
    The result of the dispatched operation based on the `resource_type`.

    Notes
    -----
    - The function loads the blob object from cloud storage with below parameters :
    - For Azure
        - 'dbutils': The dbutils object to retrive the secrets needed for the APIs.
        - 'container_name': The container where the blob object is stored.
        - 'blob_path': The local file path where the blob is present.
        - 'target_path' : The target path where the blob has to be downloaded.

    - For GCP
        - 'dbutils': The dbutils object to retrive the secrets needed for the APIs.
        - 'container_namecontainer_name': The bucket where the blob  object is stored.
        - 'blob_path': The local file path where the blob is present.
        - 'target_path' : The target path where the blob has to be downloaded.

    - It is essential to provide the correct `resource_type`. Currently supported resources are : az, gcp
    """
    resource_type = kwargs.get("resource_type", None)
    if not resource_type or resource_type in [""]:
        raise Exception("Resource type is not passed or is empty.")

    del kwargs["resource_type"]  # Delete the key since it will not be used by modules

    if resource_type not in ["az", "gcp", "azure"]:
        raise Exception(f"Uploading blob object from {resource_type} is not supported.")

    if resource_type.lower() in ["az", "azure"]:
        return __upload_blob_to_azure(**kwargs)

    if resource_type.lower() == "gcp":
        return __upload_blob_to_gcp(**kwargs)

def media_artifacts_add(cloud_provider,model_artifact_id="",entity_type=""):

    ts = int(time.time() * 1000000)

    artifacts_data = {
        "project_id": project_id,
        "version": version,
        "job_id": train_job_id,
        "run_id": train_run_id,
        "folder_type": "reports",
        "sub_folder_path": "Test_Validation",
        "media_artifacts_path": target_path,
        "model_artifact_id": model_artifact_id,
        "entity_type": entity_type
    }
    # Add additional parameters based on cloud provider
    if cloud_provider.lower() == 'azure':
        artifacts_data["container_name"] = container_name
        artifacts_data["az_storage_account"] = az_storage_account
    elif cloud_provider.lower() == 'gcp':
        artifacts_data["container_name"] = container_name
        artifacts_data["gcp_project_id"] = quota_project_id
    elif cloud_provider.lower() == 'databricks_uc':
        catalog_details = get_catalog_details(deployment_env).json().get("data")[0]
        artifacts_data["catalog_name"] = catalog_details["catalog_name"]
        artifacts_data["schema_name"] = catalog_details["catalog_schema_name"]
        artifacts_data["volume_name"] = catalog_details["volume_name"]  

    h1 = get_headers(vault_scope)
    response = requests.post(
        API_ENDPOINT + MEDIA_ARTIFACTS_ADD, json=artifacts_data, headers=h1
    )
    utils.log(
        f"\n\
    Logging task:\n\
    endpoint - {MEDIA_ARTIFACTS_ADD}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {artifacts_data}\n",
        message_run,
    )

    t = str(int(time.time() * 1000000))

    return response

# COMMAND ----------

# Fetch the model artifact details
model_artifact_details = get_model_artifacts(model_artifact_id).json()["data"][0]
print(model_artifact_details)
model_train_output_table_id = model_artifact_details["model_train_output_table_id"]
train_job_id = model_artifact_details.get("job_id", "")
train_run_id = model_artifact_details.get("run_id", "")
version = model_artifact_details.get("version", "")
if cloud_provider.lower() == "gcp":
    container_name = f"{az_container_name}_dev"
else:
    container_name = az_container_name

report_directory = f"{env}/media_artifacts/{project_id}/{version}/{train_job_id}/{train_run_id}/Test_Validation"
utils.log(f"Report Directory : {report_directory}", message_run)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Utils

# COMMAND ----------

def import_class(module_name, class_name):
    module = importlib.import_module(f"deepchecks.tabular.checks.{module_name}")
    return getattr(module, class_name)

def create_aggregated_suite(aggregated_checks):
    aggregated_suite = Suite("Custom Conditioned Checks")
    for module_name, checks in aggregated_checks.items():
        if not checks:
            print(f"\nNo checks present for {module_name}. Skipping this suite.")
            continue
        suite = Suite(module_name)
        for check in checks:
            suite.add(check)
        aggregated_suite.add(suite)
    return aggregated_suite

# COMMAND ----------

def run_tests(
    aggregated_checks, train_ds, test_ds, model, feature_columns, categorial_columns, algorithm_name
):
    # Update date_column as per expected by the model.
    if algorithm_name.lower() == "prophet":
        date_column = feature_columns[-1]
        train_ds = train_ds.withColumnRenamed(date_column, "ds")
        test_ds = test_ds.withColumnRenamed(date_column, "ds")

    # Convert dataframe to pandas -> Deepcheck works with pandas dataframe.
    train_ds_pandas = train_ds.toPandas()
    test_ds_pandas = test_ds.toPandas()

    # Prepare the DeepCheck Dataset
    train_ds_data = Dataset(
        train_ds_pandas, label=target_columns[0], cat_features=categorial_columns
    )
    test_ds_data = Dataset(
        test_ds_pandas, label=target_columns[0], cat_features=categorial_columns
    )

    # Perform full suite test on the data and the model
    evaluation_suite = create_aggregated_suite(aggregated_checks)

    # For models like prophet, the prediction is not a numpy array instead a data frame. Handle the case, by explicitly passing the predictions instead of the model.
    if algorithm_name.lower() == "prophet":
        pred_train = model.predict(train_ds_pandas[feature_columns[:-1] + ["ds"]])
        pred_test = model.predict(test_ds_pandas[feature_columns[:-1] + ["ds"]])
        test_result = evaluation_suite.run(
            train_dataset=train_ds_data,
            test_dataset=test_ds_data,
            y_pred_train=pred_train["yhat"].to_numpy(),
            y_pred_test=pred_test["yhat"].to_numpy(),
        )
    else:
        test_result = evaluation_suite.run(train_ds_data, test_ds_data, model)

    report_path = f"/dbfs/FileStore/Custom_Check_Conditions_Report_{int(time.time())}.html"

    test_result.save_as_html(report_path)

    if cloud_provider.lower() == "databricks_uc":
        catalog_details = get_catalog_details(deployment_env).json()["data"][0]
        target_path=f'/Volumes/{catalog_details["catalog_name"]}/{catalog_details["catalog_schema_name"]}/{catalog_details["volume_name"]}/{report_directory}/Custom_Check_Conditions_Report_{int(time.time())}.html'
        dbutils.fs.cp("dbfs:" + report_path.split("/dbfs")[-1], target_path)
    else:
        target_path=f"{report_directory}/Custom_Check_Conditions_Report_{int(time.time())}.html"
        upload_blob_to_cloud(
            container_name=container_name,
            source_path=report_path,
            dbutils=dbutils,
            target_path=target_path,
            resource_type=cloud_provider,
        )
    dbutils.fs.rm("dbfs:" + report_path.split("/dbfs")[-1] + ".html", True)
    utils.log(f"Successfully logged the custom check conditions report.", message_run)
    return target_path


# COMMAND ----------

try:
    # Fetch the model artifact details

    task_log_data()

    modelling_task_type = model_artifact_details.get("model_train_variables", {}).get(
        "modelling_task_type", ""
    )

    utils.log(
        f"input params :\n\
    model_artifact_details    - {model_artifact_details},\n\
    modelling_task_type       - {modelling_task_type},\n\
    ",
        message_run,
    )

    if not model_train_output_table_id or model_train_output_table_id == "":
        raise Exception(
            "model_train_output_table_id is either empty or None. Hence, the job is terminated."
        )

    # Load the data from the model_data_path
    data_to_check = uc_utils.read_data(
            spark=spark,
            sql=sql,
            dbutils=dbutils,
            vault_scope=vault_scope,
            api_endpoint=API_ENDPOINT,
            headers=h1,
            table_id=model_train_output_table_id
        )

    # Get the categorical columns
    feature_columns_to_check = (
        feature_columns
        if modelling_task_type.lower() != "forecasting"
        else feature_columns[:-1]
    )
    categorial_columns = detect_categorical_cols(
        data_to_check.select(feature_columns_to_check)
    )

    train_df_spark = data_to_check.filter(
        F.col("dataset_type_71E4E76EB8C12230B6F51EA2214BD5FE") == "train"
    ).select(feature_columns + target_columns)
    test_df_spark = data_to_check.filter(
        F.col("dataset_type_71E4E76EB8C12230B6F51EA2214BD5FE") == "test"
    ).select(feature_columns + target_columns)

    if not train_df_spark.first() and not test_df_spark.first():
        raise Exception(
            "Both train df and test df are empty. Hence, the job is terminated."
        )

    # Take random rows - Sample the data.
    train_df_spark = take_random_rows(train_df_spark)
    test_df_spark = take_random_rows(test_df_spark)

    # Perform custom conditioned tests 
    model_runtime_env_id = model_artifact_details.get("model_runtime_env_id", "1")
    job_type = model_artifact_details.get("job_type", "")
    algorithm_name = model_artifact_details.get("algorithm_name", "")
    mlflow_run_id = model_artifact_details.get("mlflow_run_id", "")
    utils.log("Trying to Load pyspark/python model...", message_run)

    ## Load model
    utils.log("Trying to Load pyspark/python/pyfunc model...", message_run)
    try:
        model = uc_utils.load_model(
            dbutils=dbutils,
            vault_scope=vault_scope,
            api_endpoint=API_ENDPOINT,
            headers=h1,
            cloud_provider=cloud_provider,
            deployment_env=deployment_env,
            model_artifact_id=model_artifact_id,
        )
    except Exception as e:
        utils.log(f"Unable to load model due to Exception : {e}", message_run)
        model = None

    # Create aggregated checks from the test config
    # Aggregate checks by module
    aggregated_checks = {
        "data_integrity": [],
        "train_test_validation": [],
        "model_evaluation": []
    }

    for check_config in test_config:
        module = check_config['model_test_module']
        check_name = check_config['check_name']

        CheckClass = import_class(module, check_name)
        check = CheckClass()

        for condition in check_config['conditions']:
            condition_name = condition['condition_name']
            params = condition["params"]
            
            # Add the condition to the check
            # check.add_condition_something(**params)
            getattr(check, f"add_condition_{condition_name}")(**params)
        
        if module in aggregated_checks:
            aggregated_checks[module].append(check)
        else:
            print("Skipping check: module not in aggregated_checks")

    target_path = run_tests(
        aggregated_checks,
        train_df_spark,
        test_df_spark,
        model,
        feature_columns,
        categorial_columns,
        algorithm_name,
    )

    Response =  media_artifacts_add(cloud_provider,model_artifact_id=model_artifact_id,entity_type="tests")

    # Update the model artifact with model test status.
    push_model_info(model_artifact_details, model_artifact_id, "success")

    # Update the status of the run to success.
    job_runs_update("success", "successful")
    job_task_update("success")
except Exception as e:
    push_model_info(model_artifact_details, model_artifact_id, "failed")
    job_runs_update("failed", f" failed with exception {str(e)}.")
    job_task_update("failed")